# SQLCoder 7B — Fine-Tuning Pipeline
Run cells one by one. Paste output in chat after each step.

In [ ]:
import sys
!{sys.executable} -m pip install -q pandas scikit-learn transformers accelerate peft datasets trl bitsandbytes plotly loguru rich tqdm
print('✅ Done')

In [ ]:
import os, json, re, csv, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

## Step 1 — Load & Inspect CSV
Put your CSV in the same folder as this notebook.

In [ ]:
CSV_PATH = 'dataset.csv'   # change if your filename differs

df = pd.read_csv(CSV_PATH)
print(f'Rows    : {len(df):,}')
print(f'Columns : {list(df.columns)}')
print(f'Nulls   :\n{df.isnull().sum()}')
print()
print('SQL TYPE DISTRIBUTION:')
for t, n in df['sql_type'].value_counts().items():
    bar = '█' * min(n // max(len(df)//50, 1), 50)
    print(f'  {t:<14} {n:>6,}  {bar}')
df.head(3)

## Step 2 — Clean & Split

In [ ]:
required = ['structured_input', 'sql', 'sql_type']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

before = len(df)
df = df.dropna(subset=required)
df['structured_input'] = df['structured_input'].astype(str).str.strip()
df['sql']              = df['sql'].astype(str).str.strip()
df['sql_type']         = df['sql_type'].astype(str).str.strip().str.upper()
df = df[(df['structured_input'].str.len() > 0) & (df['sql'].str.len() > 0)].reset_index(drop=True)
print(f'Cleaned: {before:,} → {len(df):,} rows (dropped {before-len(df)})')

# merge rare types
rare = df['sql_type'].value_counts()
rare = rare[rare < 3].index.tolist()
if rare:
    print(f'Rare types → OTHER: {rare}')
    df.loc[df['sql_type'].isin(rare), 'sql_type'] = 'OTHER'

def safe_split(data, test_size, seed, label=''):
    try:
        return train_test_split(data, test_size=test_size, stratify=data['sql_type'], random_state=seed)
    except ValueError:
        print(f'  ⚠ {label}: falling back to random split')
        return train_test_split(data, test_size=test_size, random_state=seed)

train_df, temp_df = safe_split(df, 0.20, 42, 'train/temp')
val_df,   test_df = safe_split(temp_df, 0.50, 42, 'val/test')

print(f'Train : {len(train_df):>6,} ({len(train_df)/len(df):.0%})')
print(f'Val   : {len(val_df):>6,} ({len(val_df)/len(df):.0%})')
print(f'Test  : {len(test_df):>6,} ({len(test_df)/len(df):.0%})')
print()
print(f"  {'TYPE':<14} {'TRAIN':>7} {'VAL':>7} {'TEST':>7}")
print('  ' + '-'*38)
for t in sorted(df['sql_type'].unique()):
    print(f"  {t:<14} {(train_df['sql_type']==t).sum():>7,} {(val_df['sql_type']==t).sum():>7,} {(test_df['sql_type']==t).sum():>7,}")

## Step 3 — Write JSONL Files

In [ ]:
TRAIN_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n{sql}'
INFER_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n'

def to_record(row, infer=False):
    tmpl = INFER_TMPL if infer else TRAIN_TMPL
    return {'text': tmpl.format(**row), 'sql': row['sql'], 'sql_type': row['sql_type'], 'structured_input': row['structured_input']}

def write_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  Wrote {len(records):>6,} → {path}')

write_jsonl([to_record(r)           for r in train_df.to_dict('records')], 'data/train.jsonl')
write_jsonl([to_record(r)           for r in val_df.to_dict('records')],   'data/val.jsonl')
write_jsonl([to_record(r)           for r in test_df.to_dict('records')],  'data/test.jsonl')
write_jsonl([to_record(r, infer=True) for r in test_df.to_dict('records')],'data/test_inference.jsonl')
print()
print('Sample prompt:')
print('-'*60)
print(to_record(train_df.iloc[0].to_dict())['text'])
print('-'*60)
print('✅ Step 1 done — paste output in chat')